In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm
import shutil
import json
import pickle
import time
from datetime import datetime

try:
    import xmltodict
except:
    ! pip install xmltodict

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning

from preprocessing import Preprocessing
from api import ParsePayload

#### Functions

#### Constants

In [2]:
str_splitter = '/'

# project
str_project = os.getcwd().split(str_splitter)[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split(str_splitter)[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split(str_splitter)[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

# tiers
str_tiers = """
{
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500,
}
"""
str_tiers = str_tiers.replace(' ','')

# pricing
flt_pct_threshold = 0.10
int_dollars_round_fees = 1
flt_avg_life = 2.3
flt_equity_intercept = 0.04
flt_equity_slope = 0.80
flt_securitization = 0.0595
flt_late_fee_income = 0.0042
flt_state_rate_cap = 1.0
flt_cnl_scaler = 0.8039
flt_prop_c = 0.5

Project: 20241112-simple-model-test
Task: 09_60_in_720
Subtask: 06_parser


#### Make output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Read payload

In [4]:
str_filename = 'request_8709887_4.json'
str_local_path = f'./input/{str_filename}'
try:
    dict_json_request = json.load(open(str_local_path))['request']
except KeyError:
    dict_json_request = json.load(open(str_local_path))
str_json_request = json.dumps(dict_json_request)

#### Copy preprocessing script

In [5]:
str_filename = 'preprocessing.py'
str_origin = f'../../08_prep_data/{str_filename}'
str_destination = f'./{str_filename}'
shutil.copyfile(str_origin, str_destination)

'./preprocessing.py'

#### Get preprocessing model and attributes

In [6]:
# import
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'../../08_prep_data/output/{str_filename}'
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# get attributes
flt_quantile = cls_model_preprocessing.flt_quantile
flt_income_max = cls_model_preprocessing.flt_income_max

# get imputation dictionary
str_filename = 'dict_impute.pkl'
str_local_path = f'../../10_60_in_720_test/02_model/output/{str_filename}'
dict_impute = pickle.load(open(str_local_path, 'rb'))

# get bins for scorecard
str_filename = 'dict_bins.pkl'
str_local_path = f'../../10_60_in_720_test/02_model/output/{str_filename}'
dict_bins = pickle.load(open(str_local_path, 'rb'))

#### Initialize class

In [7]:
# init
cls_model_preprocessing = Preprocessing(
    dict_impute=dict_impute,
    dict_bins=dict_bins,
    int_new_payment=600,
)
# assign
cls_model_preprocessing.flt_income_max = flt_income_max

# save
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_model_preprocessing, open(str_local_path, 'wb'))

#### Get inference model

In [8]:
str_filename = 'cls_model_inference_ml_logistic_scorecard.pkl'
str_local_path = f'../02_model/output/{str_filename}'
cls_model_inference = pickle.load(open(str_local_path, 'rb'))

#### Get list of features

In [9]:
list_cols_model = list(cls_model_inference.feature_names_in_)
list_cols_model = [f'{col.split("_binned")[0]}' for col in list_cols_model]
print(f'There are {len(list_cols_model)} features in the model')

There are 45 features in the model


#### Columns to force

In [10]:
list_cols_force = [
    'applicationdate__app',
    'fltgrossmonthly__income_sum',
    'amtfinanced__app',
    'bookvalue__app',
    'int_n_months_open__tu_pmthx',
    'int_n_months_closed__tu_pmthx',
    'int_n_months_open__tu_pmthx',
    'int_n_months_closed__tu_pmthx',
    'flt_wtd_avg_open__tu_pmthx',
    'flt_wtd_avg_closed__tu_pmthx',
    'int_bad_3mo_open__tu_pmthx',
    'int_bad_3mo_closed__tu_pmthx',
    'int_bad_6mo_open__tu_pmthx',
    'int_bad_6mo_closed__tu_pmthx',
    'int_bad_3mo_open_end__tu_pmthx',
    'int_bad_3mo_closed_end__tu_pmthx',
    'int_bad_6mo_open_end__tu_pmthx',
    'int_bad_6mo_closed_end__tu_pmthx',
    'strdealershiptrackertype__app',
    'bitdebtor__app',
    'bigaccountid__app',
    'vehicleyear__app',
    'flt_payment_open__tu_pmthx',
    'intopenbktype__app',
    'flt_avg_open__tu_pmthx',
    'flt_avg_closed__tu_pmthx',
    'totaldebt__app',
]
print(f'There are {len(list_cols_force)} columns needed for preprocessing')

There are 27 columns needed for preprocessing


#### Combine lists

In [11]:
list_cols_necessary = list_cols_model + list_cols_force
# rm dups
list_cols_necessary = list(dict.fromkeys(list_cols_necessary))
print(f'There are {len(list_cols_necessary)} columns necessary for preprocessing and prediction')

There are 69 columns necessary for preprocessing and prediction


#### Adverse action dictionary

In [12]:
df_aa = pd.read_csv('./input/df_aa.csv')
dict_aa = dict(zip(df_aa['feature'], df_aa['reason']))

# ensure captialized
dict_aa = {key: val.capitalize() for key, val in dict_aa.items()}

# pickle
str_filename = 'dict_aa.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(dict_aa, open(str_local_path, 'wb'))

#### Make sure there is an AA reason for every feature in the model

In [13]:
list_cols_model = list(cls_model_inference.feature_names_in_)
list_cols_model = [col.split('_binned')[0] for col in list_cols_model]
for a, col in enumerate(list_cols_model):
    try:
        print(f'{a+1} - {col}: {dict_aa[col]}')
    except:
        print(f'{a+1} - {col}: ERROR')
        dict_aa[col] = 'PLACEHOLDER'

1 - fltgrossmonthly__income_sum: Insufficient income
2 - au20s__tu: Insufficient credit file, length of credit
3 - miles_odometer__app: Value or type of collateral not sufficient
4 - rtl_trd__tu: Insufficient credit file, length of credit
5 - ENG-loan_to_value: Value or type of collateral not sufficient
6 - g002s__tu: Delinquent credit obligations
7 - rev322__tu: Excessive account balances
8 - balmag01__tu: Excessive account balances
9 - g232s__tu: Excessive inquiries
10 - inquirybanking12month__ln: Excessive inquiries
11 - linkt003__tu: Length of residence
12 - all235__tu: Excessive account balances
13 - ENG-wtd_avg: Derogatory payments on accounts
14 - cv15__tu: Excessive inquiries
15 - derogseverityindex__ln: Derogatory public record
16 - g407s__tu: Excessive inquiries
17 - cv13__tu: Delinquent credit obligations
18 - phoneinputproblems__ln: Unable to verify working phone number
19 - ENG-franchise: Dealer application source
20 - at24s__tu: Delinquent credit obligations
21 - jt20s__t

#### Get the LTV bins (BK)

In [14]:
# open
str_filename = 'dict_bins_ltv.pkl'
str_local_path = f'../../create_grid/01_60_in_720_bk_chargeoff/output/{str_filename}'
dict_bins_ltv_bk = pickle.load(open(str_local_path, 'rb'))

#### Get the LTV bins (no BK)

In [15]:
# open
str_filename = 'dict_bins_ltv.pkl'
str_local_path = f'../../create_grid/02_60_in_720_nobk_chargeoff/output/{str_filename}'
dict_bins_ltv_nobk = pickle.load(open(str_local_path, 'rb'))

#### Effective date - Payment History - Decline

In [16]:
str_dtm_effective_date1 = '3/11/2025 00:00:00'
dtm_effective_date1 = datetime.strptime(str_dtm_effective_date1, '%m/%d/%Y %H:%M:%S')
print(f'Effective date (payment history - decline): {dtm_effective_date1}')

Effective date (payment history - decline): 2025-03-11 00:00:00


#### Initialize class

In [17]:
cls_parser = ParsePayload(
    cls_model_preprocessing=cls_model_preprocessing,
    cls_model_inference=cls_model_inference,
    dict_aa=dict_aa,
    str_tiers=str_tiers,
    list_cols_necessary=list_cols_necessary,
    dict_bins_ltv_bk=dict_bins_ltv_bk,
    dict_bins_ltv_nobk=dict_bins_ltv_nobk,
    dtm_effective_date1=dtm_effective_date1,
)

#### Start time

In [18]:
time_start = time.perf_counter()

#### Parse payload

In [19]:
# get data
cls_parser.get_data(str_request=str_json_request)
# engineer pmt hx
cls_parser.engineer_pmt_hx()
# preprocessing
cls_parser.preprocessing()
# get predictions
cls_parser.get_predictions()
# interpolate
#cls_parser.interpolate()
# adverse action
cls_parser.adverse_action()
# counter offers
#cls_parser.counter_offers()
# generate response
cls_parser.generate_response()

Getting data...
['uniqueid__app', 'uniqueid__ln']
Application Date: 2025-03-04 08:45:59
Effective Date Payment History: 2025-03-11 00:00:00
Apply Payment History - Decline: False
[8709887107356831]: Engineering payment history...


100%|██████████| 29/29 [00:00<00:00, 2332.40it/s]

Chime: True
0    []
Name: list_pmt_hx_closed__tu_pmthx, dtype: object
0    [1, 1, 1, 1, 1, 1, 1]
Name: list_pmt_hx_open__tu_pmthx, dtype: object
0    [1, 1, 1, 1, 1, 1, 1]
Name: list_pmt_hx, dtype: object
0    7
Name: n_pmts, dtype: int64
0    1
Name: tag_min_pmts, dtype: int64
0    [1, 1, 1, 1]
Name: list_last_n_pmts, dtype: object


0    4
Name: len_last_n_pmts, dtype: int64
0    4
Name: sum_last_n_pmts, dtype: int64
0    0
Name: tag_bk, dtype: int64
0    0
Name: tag_bad_pmt_hx, dtype: int64
Bad payment history: False
[8709887107356831]: Preprocessing data...


100%|██████████| 69/69 [00:00<00:00, 5129.51it/s]


...

Binning values for scorecard...


100%|██████████| 1119/1119 [00:00<00:00, 22330.40it/s]


[8709887107356831]: Getting predictions...
[8709887107356831]: Getting adverse action...


100%|██████████| 45/45 [00:00<00:00, 4877.48it/s]
1it [00:00, 3276.80it/s]
100%|██████████| 1/1 [00:00<00:00, 26051.58it/s]

[8709887107356831]: Generating response...


#### End time

In [20]:
time_end = time.perf_counter()
flt_sec = time_end - time_start
print(f'Time to parse: {flt_sec:0.4f} sec.')

Time to parse: 0.3691 sec.


#### Show response

In [21]:
cls_parser.dict_response

{'Request_id': '',
 'Zaml_processing_id': '',
 'Response': [{'Model_name': 'gen-xiii',
   'Model_version': 'v1',
   'Results': [{'Row_id': 8709887107356831,
     'Score_pd': 0.4562623056,
     'Score_lgd': 0.2777564091,
     'Score_ecnl': 0.1267297796,
     'Score_ecnl_mod': 0.3597959828,
     'Key_factors': ['Length of residence',
      'Excessive inquiries',
      'Excessive inquiries',
      'Insufficient credit history',
      'Derogatory public record'],
     'Outlier_score': 0.0,
     'Dict_tiers': "\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"}],
   'Errors': [],
   'CounterOffers': []}]}

#### Save

In [22]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))